# lrai_locate_anything — one-click coordinated inferencer

Minimal demo. Everything heavy lives in the [`lrai_locate_anything`](https://github.com/deanofthewebb/lrai_locate_anything) package; this notebook just orchestrates:

1. Install the package from GitHub
2. Load the model, export ONNX, build TRT engines (cached on subsequent runs)
3. Single-image detection
4. Video detection on your own clip
5. (Optional) Side-by-side multi-runtime comparison + metrics
6. (Optional) TRT-LLM parallel runtime + latency comparison

**Recommended runtime:** Colab L4 (24 GB), A100, or H100. T4 finishes vision + projector engines only; LLM falls back to PyTorch.

## 1 · Install the package

In [ ]:
# ============================================================================
# Install — uses GITHUB_PAT from Colab secrets; --no-cache-dir + --force-reinstall
# guarantees we re-clone from GitHub every kernel restart (no stale pip wheel
# cache returning yesterday's build).
#
# CRITICAL: transformers >=5.0 silently leaves embed_tokens.weight and
# lm_head.weight at random init for this vendored model (the "Materializing"
# progress bar lies, no missing/unexpected/mismatched keys reported, yet weight
# std stays at initializer_range=0.02 instead of the trained 0.024). Every
# generation path then mode-collapses on <ref>$$$$$. Pinning <5.0 fixes it.
#
# We install transformers BEFORE our package and with --upgrade-strategy=eager
# so Colab's preinstalled 5.x actually gets downgraded. AFTER this cell runs,
# you MUST restart the runtime (Runtime menu > Restart) once before re-running
# the notebook — the in-memory transformers module won't downgrade itself.
# Add the PAT once via: left sidebar 🔑 Secrets → +Add → name=GITHUB_PAT.
# ============================================================================
GH_TOKEN = None
try:
    from google.colab import userdata
    for _key in ('GITHUB_PAT', 'GITHUB_TOKEN'):
        try:
            GH_TOKEN = userdata.get(_key)
            if GH_TOKEN:
                print(f'Using {_key} from Colab secrets.'); break
        except Exception:
            continue
except ImportError:
    import os
    GH_TOKEN = os.environ.get('GITHUB_PAT') or os.environ.get('GITHUB_TOKEN')

REPO = 'github.com/deanofthewebb/lrai_locate_anything.git'
URL  = f'git+https://x-access-token:{GH_TOKEN}@{REPO}' if GH_TOKEN else f'git+https://{REPO}'

# Step 1: pin transformers FIRST. --upgrade-strategy=eager forces Colab's
# preinstalled 5.x to actually downgrade to a 4.x version. Without this flag,
# pip leaves a "good enough" 5.x installed if it claims to satisfy ANYTHING.
import sys
print('Step 1/3: pinning transformers<5.0 ...')
!pip -q install --upgrade-strategy=eager "transformers>=4.55,<5.0"

# Step 2: install our package (no-deps so we don't churn Colab's scientific stack).
# Purge any pre-loaded module first so the new install gets imported clean.
for _k in [k for k in list(sys.modules) if k.startswith('lrai_locate_anything')]:
    sys.modules.pop(_k, None)
print('Step 2/3: installing lrai_locate_anything ...')
!pip -q install --no-cache-dir --force-reinstall --no-deps "{URL}"

# Step 3: detect whether transformers was downgraded in this session vs only
# on-disk (which would require a restart to take effect).
print('Step 3/3: verifying runtime state ...')
import importlib, transformers
_major = int(transformers.__version__.split('.')[0])
if _major >= 5:
    print('━' * 72)
    print('🔄  RESTART REQUIRED')
    print('━' * 72)
    print(f'  transformers {transformers.__version__} is still loaded in memory.')
    print('  The pip downgrade landed on disk but cannot take effect until the')
    print('  Python kernel is restarted.')
    print()
    print('  → Runtime menu > Restart session → then Run all from here.')
    print('━' * 72)
    raise SystemExit('Restart the runtime, then re-run this cell.')

import lrai_locate_anything as lrai, subprocess
_sha = subprocess.run(['pip', 'show', 'lrai_locate_anything'], capture_output=True, text=True).stdout
for _l in _sha.split('\n'):
    if _l.startswith(('Version', 'Location')):
        print(_l)
print(f'transformers={transformers.__version__}')

# Optional runtime deps (only the ones Colab does NOT already ship)
lrai.ensure_runtime_deps()
lrai.ensure_nvidia_stack()
print(f'lrai_locate_anything {lrai.__version__} ready.')


## 2 · Load model, export ONNX, build TRT engines

First run takes ~15-30 min (HF download + ONNX export of 4 subgraphs + TRT build).  Subsequent runs are seconds — every artefact is cached under `WORK`.

In [ ]:
from lrai_locate_anything import LocateAnythingRunner, run_image, run_video
from PIL import Image
import urllib.request, io

# Sample image determines the baked engine resolution. Pick something representative
# of your downstream workload (resolution-wise) — the engine is fixed at that size.
DEMO_IMG_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
demo_img = Image.open(io.BytesIO(urllib.request.urlopen(DEMO_IMG_URL).read())).convert('RGB')
demo_img.thumbnail((672, 672))
demo_img.save('/content/demo.jpg')

runner = LocateAnythingRunner.from_pretrained(
    auto_export=True,
    sample_image='/content/demo.jpg',
    sample_prompt='Locate all the instances that matches the following description: cat.',
)
print(f'Engines baked for grid ({runner.grid_h}, {runner.grid_w}) → frame size {runner.eng_img_w}×{runner.eng_img_h}')

## 3 · Single-image detection

In [ ]:
# Thin API call.  Diagnostics, PT auto-fallback, and the WORK/'last_inference.txt'
# dump are all built into runner.detect() (used internally by run_image).
boxes, annotated, raw = run_image(
    runner, demo_img,
    prompt='Locate all the instances that matches the following description: cat.',
)
print(f'Detected {len(boxes)} boxes:')
for b in boxes:
    print(f'  ({b[0]:.0f}, {b[1]:.0f}) → ({b[2]:.0f}, {b[3]:.0f})')
annotated


## 4 · Video detection on your own clip

Upload a video — `run_video` auto-resizes each frame to the baked engine resolution, locks the processor's `smart_resize`, and writes an annotated MP4.

In [ ]:
from pathlib import Path
CUSTOM_VID = Path('/content/custom.mp4')
OUT_VID    = Path('/content/custom_boxed.mp4')

# Colab upload widget; falls back to a path probe locally.
try:
    from google.colab import files as _gfiles
    print('Pick a video (≤200 MB).  Cell blocks until upload completes.')
    _u = _gfiles.upload()
    for name, data in _u.items():
        CUSTOM_VID.write_bytes(data)
        print(f'  saved {CUSTOM_VID}  ({len(data)/1e6:.1f} MB)')
        break
except ImportError:
    print(f'Not on Colab. Set CUSTOM_VID to a local path.')

metrics = run_video(
    runner, CUSTOM_VID, OUT_VID,
    prompt='Locate all the instances that matches the following description: roller bag</c>shoulder bag</c>carry-on</c>person.',
    max_frames=40,
)
print(f"\n{metrics['frames']} frames in {metrics['total_s']:.1f}s  "
      f"({metrics['fps']:.2f} fps, avg {metrics['avg_ms_per_frame']:.0f} ms/frame, "
      f"P99 {metrics['p99_ms']:.0f} ms)")

# Auto-download on Colab
try:
    from google.colab import files as _gfiles
    _gfiles.download(str(OUT_VID))
except Exception: pass

## 5 · (Optional) Side-by-side multi-runtime comparison

Renders each frame through up to three runtimes simultaneously: **TRT + MTP** (the fast path), **TRT (AR-only)** (`generation_mode='slow'`), and **PyTorch canonical** (if the PT model is still resident). The output video stacks the three panels horizontally for direct visual comparison.

In [ ]:
from lrai_locate_anything import run_compare
import json

metrics = run_compare(
    runner, CUSTOM_VID, '/content/compare.mp4',
    prompt='Locate all the instances that matches the following description: roller bag</c>shoulder bag</c>carry-on</c>person.',
    max_frames=20,
    include_pytorch=True,
)
print(json.dumps(metrics, indent=2))

try:
    from google.colab import files as _gfiles
    _gfiles.download('/content/compare.mp4')
except Exception: pass

## 6 · (Optional) TRT-LLM accelerator + apples-to-apples latency

Install TRT-LLM, convert the inner Qwen2 LM to TRT-LLM format, build, and benchmark prefill + decode latency vs PyTorch and plain TRT.

**Note:** TRT-LLM here runs AR-only on text (no multimodal — MTP/PBD port to spec-dec hooks is out of scope). The plain-TRT + MTP path remains the fastest end-to-end runtime for video grounding.

In [ ]:
from lrai_locate_anything.trtllm import install_trtllm, dump_qwen2_lm_only, convert_and_build, TRTLLMRunner
from lrai_locate_anything.benchmark import bench_text_runtimes, print_text_table

if install_trtllm():
    qwen_dir = dump_qwen2_lm_only(runner.model, runner.tokenizer, runner.config)
    engine_dir = convert_and_build(qwen_dir)
    trtllm = TRTLLMRunner(engine_dir, runner.tokenizer)
    print('\n=== Apples-to-apples text-only AR benchmark ===')
    results = bench_text_runtimes(runner, trtllm_runner=trtllm)
    print_text_table(results)
else:
    print('TRT-LLM unavailable on this runtime — skipping. '
          'Plain TRT + MTP is still the fastest end-to-end path in this package.')
    # Benchmark without TRT-LLM (PyTorch vs plain TRT only)
    results = bench_text_runtimes(runner, trtllm_runner=None)
    print_text_table(results)

## 7 · What you now have

- `/content/demo.jpg` — sample image
- `/content/custom_boxed.mp4` — your video with TRT+MTP boxes drawn
- `/content/compare.mp4` — (optional) 2-3 panel side-by-side runtime comparison
- `/content/locany/onnx/*.onnx` — exported ONNX subgraphs (cached for re-use)
- `/content/locany/engines/*.engine` — built TRT engines (cached)
- Latency table from §6

**To use the package outside this notebook:**
```python
from lrai_locate_anything import LocateAnythingRunner, run_image, run_video
runner = LocateAnythingRunner.from_pretrained(auto_export=True, sample_image='ref.jpg')
boxes, annotated, raw = run_image(runner, 'photo.jpg', 'Detect cats.')
```

Full module documentation: [github.com/deanofthewebb/lrai_locate_anything](https://github.com/deanofthewebb/lrai_locate_anything)